In [ ]:
# ============================================================
# AI REALIZATION GAP — EU27
# STEP 02 — AI ADOPTION FROM EUROSTAT
# Dataset: isoc_eb_ai
# ============================================================

from google.colab import drive
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import json
import itertools
from datetime import datetime

# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI_Realization_Gap_EU27"
)

ADMIN_DIR = PROJECT_ROOT / "00_admin"
RAW_DIR = PROJECT_ROOT / "01_raw_data"
INTERMEDIATE_DIR = PROJECT_ROOT / "02_intermediate_data"
PROCESSED_DIR = PROJECT_ROOT / "03_processed_data"
TABLE_DIR = PROJECT_ROOT / "05_tables"
LOG_DIR = PROJECT_ROOT / "08_logs"

for folder in [
    ADMIN_DIR,
    RAW_DIR,
    INTERMEDIATE_DIR,
    PROCESSED_DIR,
    TABLE_DIR,
    LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/AI_Realization_Gap_EU27


In [ ]:
# ============================================================
# 3. LOAD STEP 01 COUNTRY MASTER
# ============================================================

country_master_path = (
    ADMIN_DIR /
    "eu27_country_master.csv"
)

if not country_master_path.exists():
    raise FileNotFoundError(
        "eu27_country_master.csv not found. "
        "Please run Step 01 first."
    )

country_master = pd.read_csv(country_master_path)

EU27 = country_master["geo"].tolist()

print("EU-27 countries:", len(EU27))
print(EU27)

display(country_master.head())

EU-27 countries: 27
['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'EL', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']


,geo,country,iso3
0,AT,Austria,AUT
1,BE,Belgium,BEL
2,BG,Bulgaria,BGR
3,HR,Croatia,HRV
4,CY,Cyprus,CYP


In [ ]:
# ============================================================
# 4. CORE AI ADOPTION YEARS
# ============================================================

AI_YEARS = [2021, 2023, 2024, 2025]

print("AI adoption years:", AI_YEARS)

AI adoption years: [2021, 2023, 2024, 2025]


In [ ]:
# ============================================================
# 5. DOWNLOAD EUROSTAT AI DATA
# ============================================================

DATASET = "isoc_eb_ai"

url = (
    "https://ec.europa.eu/eurostat/api/dissemination/"
    f"statistics/1.0/data/{DATASET}"
    "?lang=en"
    "&sinceTimePeriod=2021"
    "&untilTimePeriod=2025"
)

print("Downloading Eurostat dataset...")
print(url)

response = requests.get(
    url,
    timeout=120
)

response.raise_for_status()

data = response.json()

print("Download completed.")
print("Dataset:", DATASET)
print("Dataset label:", data.get("label"))

https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/isoc_eb_ai?lang=en&sinceTimePeriod=2021&untilTimePeriod=2025
Download completed.
Dataset: isoc_eb_ai
Dataset label: Artificial intelligence by size class of enterprise


In [ ]:
# ============================================================
# 6. SAVE RAW EUROSTAT DATA
# ============================================================

raw_json_path = (
    RAW_DIR /
    "Step_02_Eurostat_isoc_eb_ai_raw.json"
)

with open(
    raw_json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        ensure_ascii=False
    )

print("Raw data saved:")
print(raw_json_path)

Raw data saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/01_raw_data/Step_02_Eurostat_isoc_eb_ai_raw.json


In [ ]:
# ============================================================
# 7. INSPECT EUROSTAT DIMENSIONS
# ============================================================

print("Dimension order:")
print(data["id"])

print("\nDimension sizes:")
print(data["size"])

for dim in data["id"]:

    print("\n" + "=" * 60)
    print("DIMENSION:", dim)

    dim_info = data["dimension"][dim]

    print("Label:", dim_info.get("label"))

    categories = (
        dim_info
        .get("category", {})
        .get("label", {})
    )

    # Show first categories only
    for code, label in list(categories.items())[:20]:
        print(f"{code}: {label}")

Dimension order:
['freq', 'size_emp', 'nace_r2', 'indic_is', 'unit', 'geo', 'time']

Dimension sizes:
[1, 8, 1, 63, 6, 36, 4]

DIMENSION: freq
Label: Time frequency
A: Annual

DIMENSION: size_emp
Label: Size classes in number of persons employed
0_1: From 0 to 1 person employed
0-9: From 0 to 9 persons employed
2-9: From 2 to 9 persons employed
10-49: From 10 to 49 persons employed
10-249: From 10 to 249 persons employed
GE10: 10 persons employed or more
50-249: From 50 to 249 persons employed
GE250: 250 persons employed or more

DIMENSION: nace_r2
Label: Statistical classification of economic activities in the European Community (NACE Rev. 2)
C10-S951_X_K: All activities (except agriculture, forestry and fishing, and mining and quarrying), without financial sector

DIMENSION: indic_is
Label: Information society indicator
E_AI_TTM: Enterprises using AI technologies performing analysis of written language (text mining)
E_AI_TSR: Enterprises using AI technologies converting spoken langua

In [ ]:
# ============================================================
# 8. JSON-STAT -> LONG DATAFRAME
# ============================================================

def ordered_codes(dataset, dim):
    """
    Return dimension category codes in Eurostat JSON-stat order.
    """

    index = (
        dataset["dimension"][dim]
        ["category"]["index"]
    )

    if isinstance(index, dict):

        return [
            code
            for code, position
            in sorted(
                index.items(),
                key=lambda x: x[1]
            )
        ]

    elif isinstance(index, list):
        return index

    else:
        raise TypeError(
            f"Unexpected category index type for {dim}"
        )


dimension_ids = data["id"]

dimension_codes = {
    dim: ordered_codes(data, dim)
    for dim in dimension_ids
}

for dim, codes in dimension_codes.items():
    print(dim, ":", len(codes))

freq : 1
size_emp : 8
nace_r2 : 1
indic_is : 63
unit : 6
geo : 36
time : 4


In [ ]:
# ============================================================
# 9. BUILD LONG-FORM EUROSTAT DATA
# ============================================================

combinations = list(
    itertools.product(
        *[
            dimension_codes[dim]
            for dim in dimension_ids
        ]
    )
)

print("Expected observations:", len(combinations))

values_raw = data.get("value", {})

records = []

for linear_index, combination in enumerate(combinations):

    value = None

    # Eurostat JSON may use string keys
    if str(linear_index) in values_raw:
        value = values_raw[str(linear_index)]

    # or integer keys
    elif linear_index in values_raw:
        value = values_raw[linear_index]

    record = dict(
        zip(
            dimension_ids,
            combination
        )
    )

    record["value"] = value

    records.append(record)

df_long = pd.DataFrame(records)

print("Long dataset shape:", df_long.shape)

display(df_long.head())

Expected observations: 435456
Long dataset shape: (435456, 8)


,freq,size_emp,nace_r2,indic_is,unit,geo,time,value
0,A,0_1,C10-S951_X_K,E_AI_TTM,PC_ENT,EU27_2020,2021,NaN
1,A,0_1,C10-S951_X_K,E_AI_TTM,PC_ENT,EU27_2020,2023,NaN
2,A,0_1,C10-S951_X_K,E_AI_TTM,PC_ENT,EU27_2020,2024,NaN
3,A,0_1,C10-S951_X_K,E_AI_TTM,PC_ENT,EU27_2020,2025,NaN
4,A,0_1,C10-S951_X_K,E_AI_TTM,PC_ENT,EA,2021,NaN


In [ ]:
# ============================================================
# 10. CHECK RELEVANT EUROSTAT CATEGORIES
# ============================================================

for variable in [
    "indic_is",
    "size_emp",
    "unit",
    "nace_r2"
]:

    if variable in df_long.columns:

        print("\n", "=" * 60)
        print(variable)

        print(
            sorted(
                df_long[variable]
                .dropna()
                .unique()
                .tolist()
            )
        )


indic_is
['E_AIX_CC1SIX_DA', 'E_AIX_CC1SI_DA', 'E_AIX_CC1SI_DAX', 'E_AI_ADOWN', 'E_AI_AEXT', 'E_AI_AMOWN', 'E_AI_AOS', 'E_AI_ARDY', 'E_AI_BCDP', 'E_AI_BCST', 'E_AI_BDDT', 'E_AI_BEC', 'E_AI_BIAS', 'E_AI_BIAS_ADOWN', 'E_AI_BIAS_AEXT', 'E_AI_BIAS_AMOWN', 'E_AI_BIAS_AOS', 'E_AI_BIAS_ARDY', 'E_AI_BINC', 'E_AI_BLE', 'E_AI_BLEG', 'E_AI_BNU', 'E_AI_CC', 'E_AI_CC1SIX_DA', 'E_AI_CC1SIX_DAX', 'E_AI_CC1SI_DA', 'E_AI_CC1SI_DASANY', 'E_AI_CC1SI_DASANY2', 'E_AI_CC1SI_DASGE3', 'E_AI_CC1SI_DAX', 'E_AI_CC1SI_DA_ANY', 'E_AI_DA', 'E_AI_EC', 'E_AI_P1ANY', 'E_AI_P1GE2', 'E_AI_P1GE3', 'E_AI_PBA', 'E_AI_PBAM', 'E_AI_PDI', 'E_AI_PFIN', 'E_AI_PHR', 'E_AI_PITS', 'E_AI_PLOG', 'E_AI_PME', 'E_AI_PMS', 'E_AI_PPP', 'E_AI_PRDI', 'E_AI_TANY', 'E_AI_TAR', 'E_AI_TGE2', 'E_AI_TGE3', 'E_AI_TIR', 'E_AI_TML', 'E_AI_TNLG', 'E_AI_TPA', 'E_AI_TPVSG', 'E_AI_TSR', 'E_AI_TTM', 'E_AI_TX', 'E_DI3_HI_AI_TANY', 'E_DI3_LO_AI_TANY', 'E_DI3_VHI_AI_TANY', 'E_DI3_VLO_AI_TANY']

size_emp
['0-9', '0_1', '10-249', '10-49', '2-9', '50-249', '

In [ ]:
# ============================================================
# 11. INSPECT NACE CATEGORIES
# ============================================================

nace_labels = (
    data["dimension"]["nace_r2"]
    ["category"]
    .get("label", {})
)

nace_table = pd.DataFrame({
    "nace_r2": list(nace_labels.keys()),
    "label": list(nace_labels.values())
})

display(nace_table)


,nace_r2,label
0,C10-S951_X_K,"All activities (except agriculture, forestry a..."


In [ ]:
# Find broad / total enterprise category

nace_table[
    nace_table["label"]
    .str.contains(
        "All activities|Total|All enterprises",
        case=False,
        na=False
    )
]

,nace_r2,label
0,C10-S951_X_K,"All activities (except agriculture, forestry a..."


In [ ]:
# ============================================================
# 12. IDENTIFY BROAD NACE AGGREGATE
# ============================================================

candidate_nace = nace_table[
    nace_table["label"]
    .str.contains(
        "all activities|all enterprises|total",
        case=False,
        na=False
    )
]

display(candidate_nace)

,nace_r2,label
0,C10-S951_X_K,"All activities (except agriculture, forestry a..."


In [ ]:
if len(candidate_nace) == 1:

    TOTAL_NACE = candidate_nace.iloc[0]["nace_r2"]

    print("Selected NACE aggregate:")
    print(TOTAL_NACE)

else:

    print(
        "Please inspect candidate_nace before proceeding."
    )

Selected NACE aggregate:
C10-S951_X_K


In [ ]:
# ============================================================
# 13. FILTER MAIN AI ADOPTION MEASURE
# ============================================================

ai = df_long[
    (df_long["geo"].isin(EU27)) &
    (df_long["time"].astype(str).isin(
        [str(y) for y in AI_YEARS]
    )) &
    (df_long["indic_is"] == "E_AI_TANY") &
    (df_long["size_emp"] == "GE10") &
    (df_long["unit"] == "PC_ENT") &
    (df_long["nace_r2"] == TOTAL_NACE)
].copy()

ai["year"] = ai["time"].astype(int)

ai["ai_adoption_pct"] = pd.to_numeric(
    ai["value"],
    errors="coerce"
)

ai = ai[
    [
        "geo",
        "year",
        "ai_adoption_pct"
    ]
].copy()

ai = (
    ai
    .sort_values(["geo", "year"])
    .reset_index(drop=True)
)

print("Filtered observations:", len(ai))
print("Countries:", ai["geo"].nunique())
print("Years:", sorted(ai["year"].unique()))

display(ai.head(15))

Filtered observations: 108
Countries: 27
Years: [np.int64(2021), np.int64(2023), np.int64(2024), np.int64(2025)]


,geo,year,ai_adoption_pct
0,AT,2021,8.83
1,AT,2023,10.79
2,AT,2024,20.27
3,AT,2025,29.95
4,BE,2021,10.32
5,BE,2023,13.81
6,BE,2024,24.71
7,BE,2025,34.54
8,BG,2021,3.29
9,BG,2023,3.62


In [ ]:
# ============================================================
# 14. MERGE WITH STEP 01 PANEL SKELETON
# ============================================================

skeleton_path = (
    INTERMEDIATE_DIR /
    "eu27_ai_year_panel_skeleton.csv"
)

panel = pd.read_csv(skeleton_path)

print("Skeleton shape:", panel.shape)

ai_panel = panel.merge(
    ai,
    on=["geo", "year"],
    how="left",
    validate="one_to_one"
)

print("Merged panel shape:", ai_panel.shape)

display(ai_panel.head(12))

Skeleton shape: (108, 4)
Merged panel shape: (108, 5)


,geo,country,iso3,year,ai_adoption_pct
0,AT,Austria,AUT,2021,8.83
1,AT,Austria,AUT,2023,10.79
2,AT,Austria,AUT,2024,20.27
3,AT,Austria,AUT,2025,29.95
4,BE,Belgium,BEL,2021,10.32
5,BE,Belgium,BEL,2023,13.81
6,BE,Belgium,BEL,2024,24.71
7,BE,Belgium,BEL,2025,34.54
8,BG,Bulgaria,BGR,2021,3.29
9,BG,Bulgaria,BGR,2023,3.62


In [ ]:
# ============================================================
# 15. MISSING-VALUE CHECK
# ============================================================

missing_ai = ai_panel[
    ai_panel["ai_adoption_pct"].isna()
].copy()

print(
    "Missing AI adoption observations:",
    len(missing_ai)
)

if len(missing_ai) > 0:

    display(missing_ai)

else:

    print(
        "No missing AI adoption observations."
    )

Missing AI adoption observations: 0
No missing AI adoption observations.


In [ ]:
# ============================================================
# 16. YEARLY SUMMARY
# ============================================================

year_summary = (
    ai_panel
    .groupby("year")
    ["ai_adoption_pct"]
    .agg(
        N="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max"
    )
    .reset_index()
)

display(year_summary)

,year,N,mean,median,std,min,max
0,2021,27,7.701111,7.20,5.029692,1.38,23.89
1,2023,27,8.237037,7.86,4.172594,1.51,15.17
2,2024,27,14.279259,11.31,7.177280,3.07,27.58
3,2025,27,20.729259,19.64,10.258355,5.21,42.03


In [ ]:
# ============================================================
# 17. CREATE 0-1 AI ADOPTION RATE
# ============================================================

ai_panel["ai_adoption"] = (
    ai_panel["ai_adoption_pct"] / 100
)

display(
    ai_panel[
        [
            "country",
            "year",
            "ai_adoption_pct",
            "ai_adoption"
        ]
    ].head(12)
)

,country,year,ai_adoption_pct,ai_adoption
0,Austria,2021,8.83,0.0883
1,Austria,2023,10.79,0.1079
2,Austria,2024,20.27,0.2027
3,Austria,2025,29.95,0.2995
4,Belgium,2021,10.32,0.1032
5,Belgium,2023,13.81,0.1381
6,Belgium,2024,24.71,0.2471
7,Belgium,2025,34.54,0.3454
8,Bulgaria,2021,3.29,0.0329
9,Bulgaria,2023,3.62,0.0362


In [ ]:
# ============================================================
# 18. DUPLICATE CHECK
# ============================================================

duplicates = ai_panel.duplicated(
    subset=["iso3", "year"],
    keep=False
)

print(
    "Duplicated country-year observations:",
    duplicates.sum()
)

if duplicates.any():
    display(ai_panel[duplicates])

Duplicated country-year observations: 0


In [ ]:
# ============================================================
# 19. SAVE FINAL AI ADOPTION PANEL
# ============================================================

output_path = (
    PROCESSED_DIR /
    "Step_02_AI_adoption_EU27_2021_2023_2024_2025.csv"
)

ai_panel.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(output_path)

Saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/03_processed_data/Step_02_AI_adoption_EU27_2021_2023_2024_2025.csv


In [ ]:
# ============================================================
# 20. SAVE DESCRIPTIVE TABLE
# ============================================================

summary_path = (
    TABLE_DIR /
    "Step_02_AI_adoption_year_summary.csv"
)

year_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(summary_path)

Saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/05_tables/Step_02_AI_adoption_year_summary.csv


In [ ]:
# ============================================================
# 21. QUALITY CONTROL
# ============================================================

qc_checks = {

    "Panel contains 108 country-year rows":
        len(ai_panel) == 108,

    "EU27 contains 27 countries":
        ai_panel["iso3"].nunique() == 27,

    "Four AI observation years":
        ai_panel["year"].nunique() == 4,

    "Correct years":
        set(ai_panel["year"].unique())
        == set(AI_YEARS),

    "No duplicate country-year":
        ai_panel.duplicated(
            ["iso3", "year"]
        ).sum() == 0,

    "AI adoption percentage >= 0":
        (
            ai_panel["ai_adoption_pct"]
            .dropna() >= 0
        ).all(),

    "AI adoption percentage <= 100":
        (
            ai_panel["ai_adoption_pct"]
            .dropna() <= 100
        ).all(),

    "AI adoption rate >= 0":
        (
            ai_panel["ai_adoption"]
            .dropna() >= 0
        ).all(),

    "AI adoption rate <= 1":
        (
            ai_panel["ai_adoption"]
            .dropna() <= 1
        ).all()
}

qc = pd.DataFrame({
    "check": qc_checks.keys(),
    "passed": qc_checks.values()
})

display(qc)

,check,passed
0,Panel contains 108 country-year rows,True
1,EU27 contains 27 countries,True
2,Four AI observation years,True
3,Correct years,True
4,No duplicate country-year,True
5,AI adoption percentage >= 0,True
6,AI adoption percentage <= 100,True
7,AI adoption rate >= 0,True
8,AI adoption rate <= 1,True


In [ ]:
# ============================================================
# 22. SAVE QC REPORT
# ============================================================

qc_path = (
    LOG_DIR /
    "Step_02_QC.csv"
)

qc.to_csv(
    qc_path,
    index=False,
    encoding="utf-8-sig"
)

print("QC saved:")
print(qc_path)

QC saved:
/content/drive/MyDrive/AI_Realization_Gap_EU27/08_logs/Step_02_QC.csv


In [ ]:
# ============================================================
# 23. STEP 02 FINAL REPORT
# ============================================================

print("=" * 68)
print("STEP 02 — AI ADOPTION COMPLETED")
print("=" * 68)

print(
    "Countries       :",
    ai_panel["iso3"].nunique()
)

print(
    "Years           :",
    sorted(ai_panel["year"].unique())
)

print(
    "Panel rows      :",
    len(ai_panel)
)

print(
    "Observed AI     :",
    ai_panel["ai_adoption_pct"].notna().sum()
)

print(
    "Missing AI      :",
    ai_panel["ai_adoption_pct"].isna().sum()
)

print("\nOutput:")
print(output_path)

print("=" * 68)

STEP 02 — AI ADOPTION COMPLETED
Countries       : 27
Years           : [np.int64(2021), np.int64(2023), np.int64(2024), np.int64(2025)]
Panel rows      : 108
Observed AI     : 108
Missing AI      : 0

Output:
/content/drive/MyDrive/AI_Realization_Gap_EU27/03_processed_data/Step_02_AI_adoption_EU27_2021_2023_2024_2025.csv


In [ ]:
# ============================================================
# DOWNLOAD STEP 02 OUTPUTS AS ZIP
# ============================================================

from pathlib import Path
import zipfile
from google.colab import files

step02_files = [
    PROJECT_ROOT / "01_raw_data" / "Step_02_Eurostat_isoc_eb_ai_raw.json",
    PROJECT_ROOT / "03_processed_data" / "Step_02_AI_adoption_EU27_2021_2023_2024_2025.csv",
    PROJECT_ROOT / "05_tables" / "Step_02_AI_adoption_year_summary.csv",
    PROJECT_ROOT / "08_logs" / "Step_02_QC.csv"
]

# Check that every file exists
for f in step02_files:
    print(f.name, "->", f.exists())

zip_path = Path("/content/Step_02_outputs.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in step02_files:
        if f.exists():
            z.write(f, arcname=f.name)

print("\nZIP created:", zip_path)

files.download(str(zip_path))

Step_02_Eurostat_isoc_eb_ai_raw.json -> True
Step_02_AI_adoption_EU27_2021_2023_2024_2025.csv -> True
Step_02_AI_adoption_year_summary.csv -> True
Step_02_QC.csv -> True

ZIP created: /content/Step_02_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>